In [1]:
import pandas as pd
import numpy as np

from behavioral_ml.etl import load_data
from behavioral_ml.features import mouse, key
from behavioral_ml import train, evaluation

mouse_df, key_df = load_data("../data/raw_kmt_dataset")

In [2]:
KEY_GROUPS = {
    "numpad": key.NUMPAD_KEYS,
    "tabs": key.TAB_KEYS,
    "correction": key.CORRECTION_KEYS,
    "enter": ["enter"]
}

In [3]:
FEATURES = ["Dwell Time", "Flight Time", "UU"]

# key_df_transformed = key.add_key_dynamics(key_df)
# key.aggregate_selected_key_dynamics(key_df_transformed, key_groups=KEY_GROUPS, features=FEATURES)

In [4]:
key_features = (
    key_df
    .pipe(key.add_key_dynamics)
    .pipe(key.aggregate_basic_key_dynamics)
)
key_features

Dwell_median  Dwell_iqr  Flight_median  \
User ID Is Genuine Test ID                                           
1       False      1            0.075183   0.024578       0.152125   
                   2            0.086801   0.014753       0.379757   
                   3            0.073625   0.025564       0.127848   
                   4            0.121413   0.021379       0.085070   
                   5            0.170910   0.049758       0.060691   
...                                  ...        ...            ...   
88      True       6            0.142035   0.059016       0.224558   
                   7            0.141037   0.053761       0.336086   
                   8            0.141034   0.053261       0.399101   
                   9            0.141034   0.059767       0.540136   
                   10           0.147538   0.066015       0.487123   

                            Flight_iqr  DD_median    DD_iqr  UU_median  \
User ID Is Genuine Test ID                                               
1       False      1          0.231739   0.215051  0.294208   0.216582   
                   2          0.552763   0.480338  0.689027   0.468952   
                   3          0.405288   0.206717  0.433331   0.187322   
                   4          0.805698   0.220963  0.819443   0.206442   
                   5          0.299547   0.237161  0.484073   0.184446   
...                                ...        ...       ...        ...   
88      True       6          0.871720   0.556635  0.930741   0.339085   
                   7          0.999254   0.503129  0.974750   0.475123   
                   8          0.674176   0.583142  0.811207   0.411105   
                   9          0.858222   0.705179  0.978251   0.667169   
                   10         0.644160   0.577149  0.799200   0.602152   

                              UU_iqr  
User ID Is Genuine Test ID            
1       False      1        0.320033  
                   2        0.786050  
                   3        0.421998  
                   4        1.025717  
                   5        0.218457  
...                              ...  
88      True       6        0.934233  
                   7        1.026763  
                   8        0.534146  
                   9        0.817210  
                   10       0.490124  

[1760 rows x 8 columns]

In [5]:
X, y = train.split_by_user(key_features)

In [6]:
from sklearn.metrics import roc_auc_score

metrics = {
    "roc_auc_score": roc_auc_score,
    "metrics": evaluation.get_metrics
}

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
import pprint

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(C=1),
)

y_true, y_score, results = train.train_binary_classifier(model, cv, X, y, metrics=metrics)

In [8]:
aucs = [results[user]["roc_auc_score"] for user in results]
np.mean(aucs)

np.float64(0.8401136363636365)

In [9]:
from sklearn.metrics.pairwise import manhattan_distances
from sklearn.metrics import roc_auc_score


pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

y_trues = []
y_scores = []
roc_aucs = []
for user_id in range(1, 89):
    X_enroll = X[user_id].loc[(True)].iloc[:5]
    X_eval_true = X[user_id].loc[(True)].iloc[5:]
    X_eval_false = X[user_id].loc[(False)].iloc[0:]
    X_eval = pd.concat([X_eval_true, X_eval_false])

    scaler = pipeline
    X_enroll = scaler.fit_transform(X_enroll)
    X_eval = scaler.transform(X_eval)

    template = np.median(X_enroll, axis=0, keepdims=True)

    y_score = -manhattan_distances(X_eval, template).ravel()
    y_test = 5*[1]+10*[0]
    y_test = np.array(y_test)
    
    y_trues.extend(y_test)
    y_scores.extend(y_score)

    roc_aucs.append(roc_auc_score(y_test, y_score))

np.array(roc_aucs).mean()


np.float64(0.9234090909090908)

In [10]:
X_enrolls, X_evals, y_evals = train.split_for_anomaly(X, y, enroll=5)

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

y_true = []
y_score = []
roc_aucs = []
for user_id in range(1, 89):
    X_enroll = pipeline.fit_transform(X_enrolls[user_id])
    X_eval = pipeline.transform(X_evals[user_id])

    template = np.median(X_enroll, axis=0, keepdims=True)

    score = -manhattan_distances(X_eval, template).ravel()
    
    y_true.extend(y_evals[user_id])
    y_score.extend(score)

    roc_aucs.append(roc_auc_score(y_evals[user_id], score))

np.mean(roc_aucs), np.median(roc_aucs)

(np.float64(0.9234090909090908), np.float64(0.97))

In [11]:
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

y_true, y_score = train.train_anomaly_detector(pipeline, manhattan_distances, X, y)

roc_auc_score(y_true, y_score)

0.8848631198347107

In [12]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

X_enrolls, X_evals, y_evals = train.split_for_anomaly(X, y, enroll=5)

metrics = {
    "roc_auc_score": roc_auc_score,
    "eer": evaluation.get_eer_from_scores
}

preprocessing_pipeline = make_pipeline(
    SimpleImputer(strategy="median")
)

clf = IsolationForest(n_estimators=100)

y_true = []
y_score = []
results = []
for user_id in range(1, 89):
    
    X_enroll = preprocessing_pipeline.fit_transform(X_enrolls[user_id])
    X_eval = preprocessing_pipeline.transform(X_evals[user_id])

    clf.fit(X_enroll)
    
    score = clf.decision_function(X_eval)
       
    y_true.append(y_evals[user_id])
    y_score.append(score)


    if metrics != None:
        user_results = {}
        for m in metrics:
            user_results[m] = metrics[m](y_evals[user_id], score)
            results.append(user_results)

pd.DataFrame(results).agg(["mean", "median", "std"])

,roc_auc_score,eer
mean,0.818750,0.228977
median,0.870000,0.200000
std,0.194495,0.204481


In [13]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

X_enrolls, X_evals, y_evals = train.split_for_anomaly(X, y, enroll=5)

metrics = {
    "roc_auc_score": roc_auc_score,
    "eer": evaluation.get_eer_from_scores
}

preprocessing_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

clf = OneClassSVM(kernel="rbf", gamma="auto")

y_true = []
y_score = []
results = []
for user_id in range(1, 89):
    
    X_enroll = preprocessing_pipeline.fit_transform(X_enrolls[user_id])
    X_eval = preprocessing_pipeline.transform(X_evals[user_id])

    clf.fit(X_enroll)
    
    score = clf.decision_function(X_eval)
       
    y_true.append(y_evals[user_id])
    y_score.append(score)


    if metrics != None:
        user_results = {}
        for m in metrics:
            user_results[m] = metrics[m](y_evals[user_id], score)
            results.append(user_results)

pd.DataFrame(results).agg(["mean", "median", "std"])

,roc_auc_score,eer
mean,0.930795,0.119886
median,0.970000,0.100000
std,0.094949,0.130393
